In [7]:
#andrew https://docs.google.com/document/d/1hA5mRtAiVFTnJYx3Y591MG3zVfPzRPDAdXCp0hxPiog/edit?tab=t.0

In [8]:
%pip install pandas

import pandas as pd
import numpy as np
import re

df = pd.read_csv(r"cafc-open-gouv-database-2021-01-01-to-2025-09-30-extracted-2025-10-01.csv")

def clean_data():

# Data Cleaning

#Checking  Data Structure
    #print("--------------------------------\nData Types:\n--------------------------------", df.dtypes)
    #print("--------------------------------\nRows,Columns:\n--------------------------------",df.shape)
    #print("--------------------------------\nNumber of Unique Values:\n--------------------------------",df.nunique())
    #print("--------------------------------\nName of Columns:\n--------------------------------", df.columns)
    #print("--------------------------------\nData Summary:\n--------------------------------", df.info())
    #print("--------------------------------\nFirst 5 Rows:\n--------------------------------",df.head())
#This doesn't seem useful, but included if you want
    #print("--------------------------------\nSummary:\n--------------------------------", df.describe())

#Check for null values in the dataset
    print(df.isnull().sum())

total_nulls = df.isnull().sum().sum()
print("There are " + str(total_nulls) + " null values in the dataset.")

#Rename column names, removing the French and keeping the English, and also removing spaces in the column names for easier access
df.rename(columns={
        "Date Received / Date reçue": "Date",
        "Numéro d'identification / Number ID": "ID",
        "Victim Age Range / Tranche d'âge des victimes": "Age_Range",
        "Number of Victims / Nombre de victimes": "Victims",
        "Dollar Loss /pertes financières": "Loss",
        "Complaint Received Type": "Received_Type",

        "Province/State": "Province_State",
        "Fraud and Cybercrime Thematic Categories":"Category",
        "Solicitation Method":"Method",
        "Language of Correspondence":"Language",
        "Complaint Type":"Complaint_Type"
}, inplace=True)

#print(df.head())

#Drop all columns containing information only in French
df= df.drop(
        [
            'Type de plainte reçue',
            'Pays',
            'Catégories thématiques sur la fraude et la cybercriminalité',
            'Méthode de sollicitation',
            'Langue de correspondance',
            'Type de plainte',
            'Province/État',
            'Genre'
        ]
        , axis='columns')

#print(df.head())

#Standardize text columns by stripping whitespace and converting to title case
text_cols = [
     "Received_Type",
     "Country", 
     "Province_State", 
     "Category", 
     "Gender",
     "Complaint_Type",
     "Category",
     "Method",
     "Language"]

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.title()

#Replace "Not Available" and variations for missing values with nan
df = df.replace([
    "Not Available",
    "non disponible",
    "Non disponible",
    "'Not Available / non disponible"
], np.nan)

#Fill missing values in categorical columns with "Unknown" //DISABLED
#cols = [
#     "Received_Type",
#     "Country", 
#     "Province_State", 
#     "Category", 
#     "Gender",
#     "Complaint_Type",
#     "Category",
#     "Method",
#     "Language",
#     "Gender", 
#     "Age_Range"
#     ]
#df[cols] = df[cols].fillna("Unknown")

#Replaces slashes with underscores for easier analysis
df = df.replace(r"[/]", "_", regex=True)

#Replaces spaces with underscores for easier analysis
df = df.replace(r"\s+", "_", regex=True)    

#Except for Date column, change dashes to underscores in selected string columns for easier analysis
cols_to_update = ["Received_Type", "Complaint_Type", "Country", "Province_State", "Category", "Method", "Language"]
df[cols_to_update] = df[cols_to_update].replace("-", "_", regex=True)

#print(df.head())

#Replace all instances of "Emergency (Jail, Accident, Hospital, Help)" with "Emergency" in the Category column for easier analysis
df['Category'] = df['Category'].replace('Emergency_(Jail,_Accident,_Hospital,_Help)', 'Emergency')

df["Category"].unique()


#Drop apostrophe in  Age_Range Column
df["Age_Range"] = df["Age_Range"].str[1:].fillna(df["Age_Range"])

#print(df["Age_Range"].head())

#Remove rows with non-age values in the Age_Range column
df = df[~df["Age_Range"].str.contains(r'Deceased|Décédé|Business|Entreprise|nknown', na=False, regex=True)]

df = df[~df["Age_Range"].str.contains(
    r'Deceased|Décédé|Business|Enterprise',na=False,regex=True)]


#Create a new column for the average age by applying a function to the Age_Range column
def range_to_avg(val):
        if isinstance(val, str):
            nums = re.findall(r'\d+', val)  # Extract all digits
            if len(nums) == 2:
                return (float(nums[0]) + float(nums[1])) / 2
            elif len(nums) == 1:
                return float(nums[0])
        return np.nan  # Handle non-strings or errors

df['Age_Median'] = df['Age_Range'].apply(range_to_avg)


#Remove $ and Comma  in the Loss Column
df['Loss'] = df['Loss'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
#Convert the Loss column to floating data type/numeric, coercing errors to NaN, and then fill NaN with 0
df['Loss'] = pd.to_numeric(df['Loss']).fillna(0) 

#print(df['Loss'].dtype)
#print(df['Loss'].head())

#Remove rows with unreasonable values in the Loss column
df = df[df["Loss"] >= 0]


#Drop rows where the Date column is missing
df = df.dropna(subset=["Date"])

#Convert the Date column to datetime format
df["Date"] = pd.to_datetime(df["Date"])
#print(df["Date"])

#Create a new column for the year and month of the Date Column
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
print("--------------------------------\nFirst 5 Rows:\n--------------------------------",df.head())

df.to_csv('cleaned_cafc_data.csv', index=False)
clean_data()

#Verifying  Data Structure
print("--------------------------------\nData Types:\n--------------------------------", df.dtypes)
print("--------------------------------\nRows,Columns:\n--------------------------------",df.shape)
print("--------------------------------\nNumber of Unique Values:\n--------------------------------",df.nunique())
print("--------------------------------\nName of Columns:\n--------------------------------", df.columns)
print("--------------------------------\nData Summary:\n--------------------------------", df.info())
print("--------------------------------\nFirst 5 Rows:\n--------------------------------",df.head())

Note: you may need to restart the kernel to use updated packages.
There are 0 null values in the dataset.
--------------------------------
First 5 Rows:
--------------------------------        ID       Date Received_Type Country Province_State        Category  \
0  350308 2025-09-29         Phone  Canada        Ontario   Personal_Info   
1  350309 2025-09-29         Phone  Canada         Quebec  Identity_Fraud   
2  350310 2025-09-29         Phone  Canada         Quebec       Extortion   
3  350311 2025-09-29         Phone  Canada        Ontario  Identity_Fraud   
4  350312 2025-09-29         Phone  Canada       Manitoba  Identity_Fraud   

          Method  Gender Language Age_Range Complaint_Type  Victims  Loss  \
0    Direct_Call  Female  English   30_-_39         Victim        1   0.0   
1  Other_Unknown    Male  English   50_-_59         Victim        1   0.0   
2    Direct_Call  Female   French   20_-_29        Attempt        0   0.0   
3  Other_Unknown  Female  English   40_-_49

In [9]:
print("--------------------------------\nName of Columns:\n--------------------------------", df.columns)

--------------------------------
Name of Columns:
-------------------------------- Index(['ID', 'Date', 'Received_Type', 'Country', 'Province_State', 'Category',
       'Method', 'Gender', 'Language', 'Age_Range', 'Complaint_Type',
       'Victims', 'Loss', 'Age_Median', 'Year', 'Month'],
      dtype='object')


In [10]:
#df["Country"].unique()
#df["Age_Range"].unique()
df["Complaint_Type"].unique()
#df["Province_State"].unique()
#df["Method"].unique()
#df["Gender"].unique()
#df["Age_Median"].unique()
#df["Year"].unique()
#df["Month"].unique()
#df["Received_Type"].unique()
#df["Loss"].unique()
#df["Victims"].unique()
#df["Category"].unique()
#df["Language"].unique()

array(['Victim', 'Attempt', 'Other', 'Unknown', 'Incomplete'],
      dtype=object)

In [11]:
print("--------------------------------\nFirst 5 Rows:\n--------------------------------",df.head())

--------------------------------
First 5 Rows:
--------------------------------        ID       Date Received_Type Country Province_State        Category  \
0  350308 2025-09-29         Phone  Canada        Ontario   Personal_Info   
1  350309 2025-09-29         Phone  Canada         Quebec  Identity_Fraud   
2  350310 2025-09-29         Phone  Canada         Quebec       Extortion   
3  350311 2025-09-29         Phone  Canada        Ontario  Identity_Fraud   
4  350312 2025-09-29         Phone  Canada       Manitoba  Identity_Fraud   

          Method  Gender Language Age_Range Complaint_Type  Victims  Loss  \
0    Direct_Call  Female  English   30_-_39         Victim        1   0.0   
1  Other_Unknown    Male  English   50_-_59         Victim        1   0.0   
2    Direct_Call  Female   French   20_-_29        Attempt        0   0.0   
3  Other_Unknown  Female  English   40_-_49         Victim        1   0.0   
4  Other_Unknown  Female  English   60_-_69         Victim        1   0.